# Classification and Clustering

**Objectives:**
- Preprocess real-world data for machine learning (encoding, variable selection)
- Train classification models (Logistic Regression, K-Nearest Neighbors)
- Evaluate classifiers using accuracy and confusion matrices
- Compare models and validate on held-out data
- Apply K-Means clustering and relate clusters to labels

---
## Part 1: Predicting the Credit Score

The two CSV files (source: [Kaggle](https://www.kaggle.com/datasets/clkmuhammed/creditscoreclassification?select=train.csv)) contain data about clients of a global finance company.

The goal is to predict a client's credit score category (**Good / Standard / Poor**) from available features.

We follow a **three-set approach**:
- `train.csv` → split into a **training set** and a **test set** (for model development)
- `test.csv` → kept as a **validation set** (used only at the very end to assess the chosen model)

### Step 1: Import the Data

In [ ]:
import pandas
dataset = pandas.read_csv("train.csv")
validation = pandas.read_csv("test.csv")

### Step 2: Explore the Data

Before building any model, we need to understand what the data looks like: column names, types, and distributions.

In [ ]:
dataset.columns

### Step 3: Encode the Target Variable

**Why?** The `Credit_Score` column contains text values (`Good`, `Standard`, `Poor`). Scikit-learn models require numeric labels, so we convert them to 0, 1, 2 using `LabelEncoder`.

In [ ]:
dataset['Credit_Score'].unique()

In [ ]:
from sklearn.preprocessing import LabelEncoder

cle = LabelEncoder()
dataset['Credit_Score'] = cle.fit_transform(dataset['Credit_Score'])

In [ ]:
# Check the mapping: which number corresponds to which label?
score_categories = cle.inverse_transform([0, 1, 2])
print("0 =", score_categories[0], "| 1 =", score_categories[1], "| 2 =", score_categories[2])

### Step 4: Encode Categorical Features

**Why?** Several features are stored as text (e.g., occupation, payment behaviour). We encode them as numbers so the model can use them.

We also drop the `Name` column, which is not useful for prediction.

In [ ]:
from sklearn.preprocessing import LabelEncoder as le

dataset['Payment_of_Min_Amount'] = le().fit_transform(dataset['Payment_of_Min_Amount'])
dataset['Payment_Behaviour'] = le().fit_transform(dataset['Payment_Behaviour'])
dataset['Occupation'] = le().fit_transform(dataset['Occupation'])
dataset['Type_of_Loan'] = le().fit_transform(dataset['Type_of_Loan'])
dataset['Credit_Mix'] = le().fit_transform(dataset['Credit_Mix'])

In [ ]:
# Drop the Name column — it has no predictive value
dataset = dataset.drop(columns=["Name"], errors='ignore')

### Step 5: Visualize the Data

Plots help us understand distributions and relationships between variables before modelling.

In [ ]:
import seaborn as sns
sns.histplot(dataset['Credit_Score'])

> **Interpretation:** The three credit score categories are not perfectly balanced, but none is extremely rare. This is good — a heavily imbalanced dataset would require special handling.

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(14, 10))
sns.heatmap(dataset.select_dtypes(include='number').corr())
plt.title("Correlation heatmap")
plt.show()

> **Interpretation:** ID, Customer_ID, SSN are not correlated with other variables (as expected — they are identifiers). `Occupation` also shows low correlation because LabelEncoder assigned arbitrary numbers to categories. A nonlinear model can still use it; for a linear model, one-hot encoding would be better.

---
### Step 6: Split Into Training and Test Sets

We split `dataset` into 75% training / 25% test.

In [ ]:
import sklearn.model_selection

df_train, df_test = sklearn.model_selection.train_test_split(
    dataset, test_size=0.25, random_state=243
)

### Step 7: Select Features and Prepare X / Y

**Why select variables?** Not all columns are useful features. IDs, SSNs, and similar identifiers would add noise. We keep only meaningful predictors.

In [ ]:
variables = [
    'Changed_Credit_Limit',
    'Payment_of_Min_Amount',
    'Credit_Mix',
    'Delay_from_due_date',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Age',
    'Monthly_Balance',
    'Num_of_Delayed_Payment',
    'Outstanding_Debt',
    'Payment_Behaviour',
    'Credit_History_Age',
    'Num_Bank_Accounts',
    'Credit_Utilization_Ratio'
]

In [ ]:
# Features and labels for the training set
X = df_train[variables]
Y = df_train['Credit_Score']

# Features and labels for the test set
X_test = df_test[variables]
Y_test = df_test['Credit_Score']

---
### Step 8: Logistic Regression

**What is logistic regression?** It is the classification counterpart of linear regression. Instead of predicting a continuous value, it predicts the *probability* of belonging to each category.

We increase `max_iter` because the default (100) may not be enough for convergence.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X, Y)

In [ ]:
print("Training accuracy:", model_lr.score(X, Y))
print("Test accuracy:    ", model_lr.score(X_test, Y_test))

> **Interpretation:** The accuracy tells us the proportion of correctly classified observations. Training and test accuracy are similar, so the model is not overfitting — but overall accuracy may be modest.

---
### Step 9: Confusion Matrix

**Why a confusion matrix?** Accuracy alone can be misleading, especially with imbalanced classes. The confusion matrix shows us *which* categories are correctly predicted and *where* the model makes mistakes.

In [ ]:
actual = Y_test
predicted = model_lr.predict(X_test)

In [ ]:
from sklearn import metrics

confusion_matrix = metrics.confusion_matrix(actual, predicted)
confusion_matrix

In [ ]:
cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_lr.plot(ax=ax)
plt.title("Logistic Regression — Confusion Matrix")
plt.show()

#### Normalized Confusion Matrices

We can normalize the confusion matrix in two ways:
- **By row** (divide each row by its total): shows what fraction of *actual* labels were predicted correctly → measures **recall / sensitivity**
- **By column** (divide each column by its total): shows what fraction of *predicted* labels were actually correct → measures **precision**

In [ ]:
# Normalize by row (recall): what % of each actual category was correctly detected?
cm_by_row = confusion_matrix / confusion_matrix.sum(axis=1)[:, None] * 100
cm_by_row

> **Interpretation:** Look at the diagonal values. For example, if the "Poor" row shows ~33% on the diagonal, it means only 33% of actual "Poor" clients were correctly identified — the rest were misclassified.

In [ ]:
cm_display_row = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_row, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_row.plot(ax=ax)
plt.title("Logistic Regression — Normalized by Row (Recall)")
plt.show()

In [ ]:
# Normalize by column (precision): of all predictions for a category, how many were correct?
cm_by_col = confusion_matrix / confusion_matrix.sum(axis=0)[None, :] * 100
cm_by_col

In [ ]:
cm_display_col = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_col, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_col.plot(ax=ax)
plt.title("Logistic Regression — Normalized by Column (Precision)")
plt.show()

> **Interpretation:** If the "Poor" column shows ~54% on the diagonal, it means that of all clients *predicted* as "Poor", only 54% actually were — the rest were false alarms.

---
### Step 10: K-Nearest Neighbors (KNN)

**Why try another model?** Logistic regression is linear — it may miss nonlinear patterns. KNN is a simple nonlinear classifier: it predicts a label based on the majority vote of the $k$ closest training points.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier()
model_knn.fit(X, Y)

In [ ]:
actual = Y_test
predicted = model_knn.predict(X_test)

In [ ]:
confusion_matrix_knn = metrics.confusion_matrix(actual, predicted)
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn, display_labels=score_categories
)

#### Side-by-Side Comparison

Let's compare both models visually.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
ax.grid(False)
cm_display_lr.plot(ax=ax)
ax.set_title("Logistic Regression")

ax = axes[1]
ax.grid(False)
cm_display_knn.plot(ax=ax)
ax.set_title("KNN Classifier")

plt.tight_layout()
plt.show()

> **Interpretation:** The KNN classifier performs noticeably better — the diagonal values (correct predictions) are larger and the off-diagonal values (errors) are smaller. This suggests that the relationship between features and credit score is **nonlinear**, which KNN can capture but logistic regression cannot.

---
### Step 11: Validate on the Held-Out Set

**Why?** We chose KNN because it performed better *on the test set*. But since the test set influenced our model choice, it is no longer a truly independent evaluation. The **validation set** (`test.csv`) gives us an unbiased estimate of real-world performance.

We must preprocess the validation set in exactly the same way as the training data.

> **Note:** `LabelEncoder` assigns numbers in alphabetical order, so applying it separately to two datasets produces the same mapping — as long as both datasets contain the same categories.

In [ ]:
from sklearn.preprocessing import LabelEncoder

validation['Payment_of_Min_Amount'] = LabelEncoder().fit_transform(validation['Payment_of_Min_Amount'])
validation['Payment_Behaviour'] = LabelEncoder().fit_transform(validation['Payment_Behaviour'])
validation['Occupation'] = LabelEncoder().fit_transform(validation['Occupation'])
validation['Type_of_Loan'] = LabelEncoder().fit_transform(validation['Type_of_Loan'])
validation['Credit_Mix'] = LabelEncoder().fit_transform(validation['Credit_Mix'])
validation['Credit_Score'] = LabelEncoder().fit_transform(validation['Credit_Score'])

In [ ]:
X_valid = validation[variables]
Y_valid = validation['Credit_Score']

In [ ]:
actual_valid = Y_valid
predicted_valid = model_knn.predict(X_valid)

In [ ]:
confusion_matrix_valid = metrics.confusion_matrix(actual_valid, predicted_valid)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=0)[None, :]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalized by Column (Precision)")

ax = axes[1]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=1)[:, None]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalized by Row (Recall)")

plt.tight_layout()
plt.show()

> **Interpretation:** The validation results are close to the test set results — this confirms that KNN generalizes well and our model choice was sound. About 73% of "Poor" ratings are correctly detected (recall), and about 75% of predicted "Poor" ratings are accurate (precision).

---
## Part 2: Segmenting the Bank Clients (K-Means Clustering)

**What is clustering?** Unlike classification (supervised), clustering is **unsupervised** — we do not give the model any labels. It groups observations into clusters based on similarity alone.

**Goal:** Can we find natural groups among clients *without* using the credit score? And if so, do these groups relate to credit quality?

In [ ]:
from sklearn.cluster import KMeans

km_model = KMeans(n_clusters=3)
km_model.fit(dataset.select_dtypes(include='number'))

In [ ]:
# Assign each client to a cluster
dataset['cluster'] = km_model.predict(dataset.select_dtypes(include='number'))

### Are the Clusters Related to the Credit Score?

In [ ]:
dataset.groupby('cluster')['Credit_Score'].value_counts(normalize=True)

> **Interpretation:** The proportion of each credit score category is roughly the same across clusters. This suggests that the clusters capture other dimensions of client similarity (e.g., income level, region) that are **not strongly correlated** with the ability to repay. In other words, credit quality is not the main axis along which clients differ.